# 🎙️ Deepfake Audio Detection System
### End-to-End Pipeline using the Fake-or-Real (FoR) Dataset

**Pipeline Phases:**
1. Dataset Setup
2. Data Preprocessing & Feature Extraction
3. Model Building (SVM / Random Forest / XGBoost)
4. Model Selection & Saving
5. Gradio Web App
6. Performance Optimization

---
## PHASE 1 — Dataset Setup

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1.1 — Install Required Libraries
# ─────────────────────────────────────────────────────────────
!pip install -q librosa numpy pandas scikit-learn gradio joblib xgboost tqdm kaggle
print("✅ All libraries installed successfully.")

✅ All libraries installed successfully.


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 1.2 — Configure Kaggle API Credentials
#
# Option A (recommended): Upload kaggle.json from your machine
# Option B: Enter credentials manually below
# ─────────────────────────────────────────────────────────────
import os
from google.colab import files

USE_MANUAL_CREDENTIALS = False   # ← Set True to type them inline

if USE_MANUAL_CREDENTIALS:
    KAGGLE_USERNAME = "your_kaggle_username"   # ← edit
    KAGGLE_KEY      = "your_kaggle_api_key"    # ← edit
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        import json
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle credentials set manually.")
else:
    # Upload kaggle.json from your local machine
    print("📂 Please upload your kaggle.json file ...")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ kaggle.json uploaded and configured.")

📂 Please upload your kaggle.json file ...


Saving kaggle.json to kaggle.json
✅ kaggle.json uploaded and configured.


In [3]:
# ── CELL 1.3 (re-download) ──────────────────────────────────
import os

DATASET_DIR  = "/content/for_dataset"
DATASET_SLUG = "mohammedabdeldayem/the-fake-or-real-dataset"

os.makedirs(DATASET_DIR, exist_ok=True)
!kaggle datasets download -d {DATASET_SLUG} -p {DATASET_DIR} --unzip
print("✅ Done")

# Show what extracted
for root, dirs, files in os.walk(DATASET_DIR):
    depth = root.replace(DATASET_DIR, "").count(os.sep)
    if depth > 4: continue
    print("  " * depth + os.path.basename(root) + "/")
    wav = [f for f in files if f.endswith(".wav")]
    if wav:
        print("  " * (depth+1) + f"({len(wav)} wav files)")

Dataset URL: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset
License(s): GNU Lesser General Public License 3.0
100% 16.0G/16.0G [02:27<00:00, 117MB/s]

User cancelled operation
✅ Done
for_dataset/
  for-2sec/
    for-2seconds/
      testing/
        fake/
          (544 wav files)
        real/
          (544 wav files)
      validation/
        fake/
          (1413 wav files)
        real/
          (1413 wav files)
      training/
        fake/
          (6978 wav files)
        real/
          (6978 wav files)
  for-original/
    for-original/
      testing/
        fake/
          (2370 wav files)
        real/
          (2264 wav files)
      training/
        fake/
          (3263 wav files)
        real/
          (7942 wav files)
  for-norm/
    for-norm/
      testing/
        fake/
          (2370 wav files)
        real/
          (2264 wav files)
      validation/
        fake/
          (5398 wav files)
        real/
          (5400 wav files)


In [4]:
# ── DIAGNOSTIC: check file extensions by label folder
import os

for root, dirs, files in os.walk("/content/for_dataset"):
    folder = os.path.basename(root).lower()
    if folder in ("real", "fake"):
        ext_counts = {}
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            ext_counts[ext] = ext_counts.get(ext, 0) + 1
        print(f"{folder.upper():>5}  {root}")
        for ext, count in sorted(ext_counts.items()):
            print(f"         {ext:>5} → {count:,} files")
        print()

 FAKE  /content/for_dataset/for-2sec/for-2seconds/testing/fake
          .wav → 544 files

 REAL  /content/for_dataset/for-2sec/for-2seconds/testing/real
          .wav → 544 files

 FAKE  /content/for_dataset/for-2sec/for-2seconds/validation/fake
          .wav → 1,413 files

 REAL  /content/for_dataset/for-2sec/for-2seconds/validation/real
          .wav → 1,413 files

 FAKE  /content/for_dataset/for-2sec/for-2seconds/training/fake
          .wav → 6,978 files

 REAL  /content/for_dataset/for-2sec/for-2seconds/training/real
          .wav → 6,978 files

 FAKE  /content/for_dataset/for-original/for-original/testing/fake
          .wav → 2,370 files

 REAL  /content/for_dataset/for-original/for-original/testing/real
          .wav → 2,264 files

 FAKE  /content/for_dataset/for-original/for-original/training/fake
          .mp3 → 23,678 files
          .wav → 3,263 files

 REAL  /content/for_dataset/for-original/for-original/training/real
          .wav → 7,942 files

 FAKE  /content/fo

In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 1.4 (FINAL FIXED) — correct hardcoded path
# ─────────────────────────────────────────────────────────────
import os, pandas as pd

FOR_NORM_DIR = "/content/for_dataset/for-norm/for-norm"   # confirmed correct
label_map    = {"real": 0, "fake": 1}
split_names  = {"training", "validation", "testing"}

def scan_dataset(base_dir):
    records = []
    for root, dirs, files in os.walk(base_dir):
        folder = os.path.basename(root).lower()
        if folder not in label_map:
            continue
        cls_label = label_map[folder]
        split = "unknown"
        for part in root.replace("\\", "/").split("/"):
            if part.lower() in split_names:
                split = part.lower()
                break
        for fname in files:
            if fname.lower().endswith(".wav"):
                records.append({
                    "file_path": os.path.join(root, fname),
                    "label":     cls_label,
                    "split":     split,
                })
    return pd.DataFrame(records)

df = scan_dataset(FOR_NORM_DIR)

print(f"📊 Total samples : {len(df):,}")
print(f"\n📊 Class distribution:")
print(df["label"].value_counts().rename({0: "real", 1: "fake"}).to_string())
print(f"\n📊 Split distribution:")
print(df["split"].value_counts().to_string())
print(f"\n🔍 Sample paths:")
print(df["file_path"].sample(3, random_state=42).to_string(index=False))

📊 Total samples : 69,300

📊 Class distribution:
label
fake    34695
real    34605

📊 Split distribution:
split
training      53868
validation    10798
testing        4634

🔍 Sample paths:
/content/for_dataset/for-norm/for-norm/training...
/content/for_dataset/for-norm/for-norm/training...
/content/for_dataset/for-norm/for-norm/training...


In [6]:
# ─────────────────────────────────────────────────────────────
# CELL 1.5 — Diagnose actual folder structure
# Run this FIRST to see what the zip actually extracted to
# ─────────────────────────────────────────────────────────────
import os

def print_tree(root, max_depth=4, _prefix="", _depth=0):
    if _depth > max_depth:
        return
    try:
        entries = sorted(os.listdir(root))
    except PermissionError:
        return
    for i, name in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        path = os.path.join(root, name)
        print(_prefix + connector + name)
        if os.path.isdir(path):
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(path, max_depth, _prefix + extension, _depth + 1)

print(f"📁 Full tree under: {DATASET_DIR}\n")
print_tree(DATASET_DIR, max_depth=4)

Streaming output truncated to the last 5000 lines.
│               ├── file13406.wav
│               ├── file13407.wav
│               ├── file13409.wav
│               ├── file1341.wav
│               ├── file13410.wav
│               ├── file13411.wav
│               ├── file13412.wav
│               ├── file13413.wav
│               ├── file13414.wav
│               ├── file13415.wav
│               ├── file13416.wav
│               ├── file13417.wav
│               ├── file13419.wav
│               ├── file1342.wav
│               ├── file13420.wav
│               ├── file13421.wav
│               ├── file13422.wav
│               ├── file13423.wav
│               ├── file13424.wav
│               ├── file13425.wav
│               ├── file13426.wav
│               ├── file13428.wav
│               ├── file13429.wav
│               ├── file1343.wav
│               ├── file13430.wav
│               ├── file13431.wav
│               ├── file13432.wav
│               ├── file13433.wav


---
## PHASE 2 — Data Preprocessing & Feature Extraction

In [7]:
import numpy as np, os

SAMPLE_RATE  = 22050
DURATION     = 2.0
MAX_SAMPLES  = int(SAMPLE_RATE * DURATION)
N_MFCC       = 40
SUBSET_SIZE  = 10_000
TEST_SIZE    = 0.20
RANDOM_STATE = 42
N_JOBS       = -1

ARTIFACT_DIR    = "/content/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

SCALER_PATH     = os.path.join(ARTIFACT_DIR, "scaler.pkl")
BEST_MODEL_PATH = os.path.join(ARTIFACT_DIR, "best_model.pkl")

print("✅ Config loaded")
print(f"   Sample rate : {SAMPLE_RATE} Hz")
print(f"   Duration    : {DURATION} s")
print(f"   MFCC coeffs : {N_MFCC}")

✅ Config loaded
   Sample rate : 22050 Hz
   Duration    : 2.0 s
   MFCC coeffs : 40


In [8]:
import librosa, warnings
import numpy as np
warnings.filterwarnings("ignore")

def load_audio(file_path, sr=SAMPLE_RATE, max_samples=MAX_SAMPLES):
    try:
        y, _ = librosa.load(file_path, sr=sr, duration=DURATION, mono=True)
        if len(y) < max_samples:
            y = np.pad(y, (0, max_samples - len(y)), mode="constant")
        else:
            y = y[:max_samples]
        return y
    except Exception:
        return None


def extract_features(file_path, sr=SAMPLE_RATE):
    """
    Enhanced 193-dim feature vector:
      MFCC         40 mean + 40 std  = 80
      Delta MFCC   40 mean + 40 std  = 80  ← captures temporal change
      Chroma       12 mean + 12 std  = 24
      Contrast      7 mean +  7 std  = 14  (kept — good for voice vs TTS)
      ZCR           1 mean +  1 std  =  2
      RMS energy    1 mean +  1 std  =  2  ← TTS has different energy envelope
      Mel spec     20 mean + 20 std  = 40  ← low-level spectral texture
      Pitch (f0)    1 mean +  1 std  =  2  ← real voices have natural pitch variation
                                    Total = 244
    """
    y = load_audio(file_path, sr=sr)
    if y is None:
        return None

    # MFCC + deltas
    mfcc       = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    delta_mfcc = librosa.feature.delta(mfcc)

    # Chroma
    chroma     = librosa.feature.chroma_stft(y=y, sr=sr)

    # Spectral contrast
    contrast   = librosa.feature.spectral_contrast(y=y, sr=sr)

    # ZCR
    zcr        = librosa.feature.zero_crossing_rate(y)

    # RMS energy
    rms        = librosa.feature.rms(y=y)

    # Mel spectrogram (20 bands)
    mel        = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=20)
    mel_db     = librosa.power_to_db(mel, ref=np.max)

    # Fundamental frequency (pitch) — real voices have natural pitch variation
    # TTS voices often have unnaturally smooth pitch
    f0, _, _   = librosa.pyin(y,
                               fmin=librosa.note_to_hz("C2"),
                               fmax=librosa.note_to_hz("C7"),
                               sr=sr)
    f0_clean   = f0[~np.isnan(f0)]
    pitch_mean = float(np.mean(f0_clean)) if len(f0_clean) > 0 else 0.0
    pitch_std  = float(np.std(f0_clean))  if len(f0_clean) > 0 else 0.0

    return np.concatenate([
        mfcc.mean(axis=1),       mfcc.std(axis=1),
        delta_mfcc.mean(axis=1), delta_mfcc.std(axis=1),
        chroma.mean(axis=1),     chroma.std(axis=1),
        contrast.mean(axis=1),   contrast.std(axis=1),
        [zcr.mean(),             zcr.std()],
        [rms.mean(),             rms.std()],
        mel_db.mean(axis=1),     mel_db.std(axis=1),
        [pitch_mean,             pitch_std],
    ])


# Sanity check
_f = extract_features(df["file_path"].iloc[0])
print(f"✅ Feature extractor ready — vector length: {len(_f)}")

# Verify it works on a real and fake file
for lval, lname in [(0, "real"), (1, "fake")]:
    sample = df[df["label"] == lval].iloc[0]
    feat   = extract_features(sample["file_path"])
    print(f"   {lname}: {feat.shape[0]} features — "
          f"min={feat.min():.3f} max={feat.max():.3f} "
          f"({'OK' if not np.any(np.isnan(feat)) else 'NaN FOUND'})")

KeyboardInterrupt: 

In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import numpy as np, os

SAMPLES_PER_CLASS = {
    "training":   1000,
    "validation":  1000,
    "testing":     1000,
}

def extract_chunked(file_paths, labels, tag="", chunk_size=200):
    # chunk_size reduced to 200 because pyin() is slower than mfcc
    all_feats, all_labels = [], []
    chunks = list(range(0, len(file_paths), chunk_size))
    for i, start in enumerate(chunks):
        end     = min(start + chunk_size, len(file_paths))
        results = Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(extract_features)(fp)
            for fp in tqdm(file_paths[start:end],
                           desc=f"{tag} chunk {i+1}/{len(chunks)}",
                           leave=False)
        )
        for feat, lbl in zip(results, labels[start:end]):
            if feat is not None:
                all_feats.append(feat)
                all_labels.append(lbl)
        np.save(os.path.join(ARTIFACT_DIR, f"X_{tag}.npy"),
                np.array(all_feats, dtype=np.float32))
        np.save(os.path.join(ARTIFACT_DIR, f"y_{tag}.npy"),
                np.array(all_labels, dtype=np.int8))
        print(f"   [{tag}] chunk {i+1}/{len(chunks)}: {len(all_feats):,} saved")
    return (np.array(all_feats, dtype=np.float32),
            np.array(all_labels, dtype=np.int8))


splits_data = {}
for split_name, n_per_class in SAMPLES_PER_CLASS.items():
    sdf = df[df["split"] == split_name].copy()
    if n_per_class is None:
        sampled = sdf.reset_index(drop=True)
    else:
        sampled = (
            sdf.groupby("label", group_keys=False)
               .apply(lambda g: g.sample(min(n_per_class, len(g)),
                                         random_state=RANDOM_STATE))
               .reset_index(drop=True)
        )
    real_n = (sampled.label == 0).sum()
    fake_n = (sampled.label == 1).sum()
    print(f"\n⚙️  {split_name}: {real_n} real + {fake_n} fake")
    X, y = extract_chunked(
        sampled["file_path"].tolist(),
        sampled["label"].tolist(),
        tag=split_name
    )
    splits_data[split_name] = (X, y)

X_train, y_train = splits_data["training"]
X_val,   y_val   = splits_data["validation"]
X_test,  y_test  = splits_data["testing"]

print(f"\n✅ Train : {X_train.shape}")
print(f"✅ Val   : {X_val.shape}")
print(f"✅ Test  : {X_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

joblib.dump(scaler, SCALER_PATH)

print("✅ Scaler fitted")
print(f"   Train : {X_train.shape} | {(y_train==0).sum()} real / {(y_train==1).sum()} fake")
print(f"   Val   : {X_val.shape}")
print(f"   Test  : {X_test.shape}")

---
## PHASE 3 — Model Building

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te,
                   do_cv=False, cv_folds=5):
    print(f"\n{'='*55}")
    print(f"  Training : {name}")
    print(f"{'='*55}")

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_te)
    else:
        y_prob = y_pred

    metrics = {
        "Model"    : name,
        "Accuracy" : accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall"   : recall_score(y_te, y_pred),
        "F1"       : f1_score(y_te, y_pred),
        "ROC-AUC"  : roc_auc_score(y_te, y_prob),
    }

    for k, v in metrics.items():
        if k != "Model":
            print(f"   {k:<12}: {v:.4f}")

    print(f"\n{classification_report(y_te, y_pred, target_names=['Real','Fake'])}")

    if do_cv:
        from sklearn.model_selection import cross_val_score
        import numpy as np
        cv_scores = cross_val_score(
            model, np.vstack([X_tr, X_te]),
            np.concatenate([y_tr, y_te]),
            cv=cv_folds, scoring="f1", n_jobs=N_JOBS
        )
        print(f"   CV F1 ({cv_folds}-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        metrics["CV_F1_mean"] = cv_scores.mean()
        metrics["CV_F1_std"]  = cv_scores.std()

    metrics["_model_obj"] = model
    return metrics

print("✅ Evaluation helper ready")

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes = (256, 128, 64),  # 3 hidden layers
    activation         = "relu",
    solver             = "adam",
    alpha              = 0.001,           # L2 regularization
    batch_size         = 256,
    learning_rate      = "adaptive",
    max_iter           = 100,
    early_stopping     = True,            # stops if val score doesn't improve
    validation_fraction= 0.1,
    n_iter_no_change   = 10,
    random_state       = RANDOM_STATE,
    verbose            = True
)

mlp_metrics = evaluate_model(
    "MLP Neural Net", mlp_model,
    X_train, y_train,
    X_test,  y_test,
    do_cv=False
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators     = 200,
    max_depth        = None,
    min_samples_leaf = 1,
    class_weight     = "balanced",
    random_state     = RANDOM_STATE,
    n_jobs           = N_JOBS
)

rf_metrics = evaluate_model(
    "Random Forest", rf_model,
    X_train, y_train,
    X_test,  y_test,
    do_cv=False
)

In [ ]:
from xgboost import XGBClassifier

_neg = int((y_train == 0).sum())
_pos = int((y_train == 1).sum())

xgb_model = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 6,
    learning_rate     = 0.1,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = _neg / _pos,
    use_label_encoder = False,
    eval_metric       = "logloss",
    random_state      = RANDOM_STATE,
    n_jobs            = N_JOBS,
    device            = "cuda"
)

xgb_metrics = evaluate_model(
    "XGBoost", xgb_model,
    X_train, y_train,
    X_test,  y_test,
    do_cv=False
)

---
## PHASE 4 — Model Selection & Saving

In [ ]:
import joblib, pandas as pd

all_metrics = [mlp_metrics, rf_metrics, xgb_metrics]

comparison_df = pd.DataFrame([
    {k: v for k, v in m.items() if k != "_model_obj"}
    for m in all_metrics
]).set_index("Model").sort_values(["F1", "ROC-AUC"], ascending=False)

print("📊 Model Comparison:")
print(comparison_df.to_string(float_format="{:.4f}".format))

best_name  = comparison_df.index[0]
best_model = next(m["_model_obj"] for m in all_metrics if m["Model"] == best_name)
best_row   = comparison_df.iloc[0]

print(f"\n🏆 Best model : {best_name}")
print(f"   F1         : {best_row['F1']:.4f}")
print(f"   ROC-AUC    : {best_row['ROC-AUC']:.4f}")

joblib.dump(best_model, BEST_MODEL_PATH)
print(f"\n💾 Saved → {BEST_MODEL_PATH}")
print(f"💾 Saved → {SCALER_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Mount Google Drive and save all artifacts
# ─────────────────────────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

# Create a dedicated folder in your Drive
DRIVE_SAVE_DIR = "/content/drive/MyDrive/deepfake_audio_model"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Files to save
artifacts = {
    "best_model.pkl" : BEST_MODEL_PATH,
    "scaler.pkl"     : SCALER_PATH,
    "X_training.npy" : os.path.join(ARTIFACT_DIR, "X_training.npy"),
    "y_training.npy" : os.path.join(ARTIFACT_DIR, "y_training.npy"),
    "X_testing.npy"  : os.path.join(ARTIFACT_DIR, "X_testing.npy"),
    "y_testing.npy"  : os.path.join(ARTIFACT_DIR, "y_testing.npy"),
}

print("💾 Saving to Google Drive...\n")
for fname, src_path in artifacts.items():
    dst = os.path.join(DRIVE_SAVE_DIR, fname)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"   ✅ {fname:<25} → {size_mb:.1f} MB")
    else:
        print(f"   ⚠️  {fname:<25} → not found, skipping")

print(f"\n✅ All artifacts saved to: {DRIVE_SAVE_DIR}")

---
## PHASE 5 — Gradio Web App

In [ ]:
import joblib, numpy as np

_model  = joblib.load(BEST_MODEL_PATH)
_scaler = joblib.load(SCALER_PATH)

assert hasattr(_scaler, "mean_"), "❌ Scaler not fitted"
print(f"✅ Model  : {type(_model).__name__}")
print(f"✅ Scaler : fitted on {_scaler.mean_.shape[0]} features")

def predict_audio(audio_path):
    if audio_path is None:
        return "ERROR", 0.0, 0.0
    feat = extract_features(audio_path)
    if feat is None:
        return "ERROR", 0.0, 0.0
    feat_scaled = _scaler.transform(feat.reshape(1, -1))
    proba       = _model.predict_proba(feat_scaled)[0]
    prob_real   = float(proba[0]) * 100
    prob_fake   = float(proba[1]) * 100
    verdict     = "FAKE" if prob_fake > prob_real else "REAL"
    confidence  = prob_fake if verdict == "FAKE" else prob_real
    return verdict, round(confidence, 2), round(prob_fake, 2)

# Sanity check
print("\n🔬 Sanity check:\n")
print(f"{'True':>5}  {'Pred':>5}  {'Conf%':>7}  {'Fake%':>7}  OK?")
print("─" * 40)
correct = 0
for lval, lname in [(0, "real"), (1, "fake")]:
    for _, row in df[df["label"] == lval].sample(3, random_state=7).iterrows():
        verdict, conf, fake_p = predict_audio(row["file_path"])
        true_str = "REAL" if lval == 0 else "FAKE"
        ok = "✅" if verdict == true_str else "❌"
        if verdict == true_str: correct += 1
        print(f"{true_str:>5}  {verdict:>5}  {conf:>6.1f}%  {fake_p:>6.1f}%  {ok}")
print(f"\nScore: {correct}/6")

In [ ]:
import gradio as gr

def gradio_predict(audio):
    if audio is None:
        return "⚠️ Please upload an audio file.", 0.0, 0.0
    verdict, confidence, fake_prob = predict_audio(audio)
    if verdict == "ERROR":
        return "❌ Could not process this file.", 0.0, 0.0
    emoji  = "🔴" if verdict == "FAKE" else "🟢"
    result = f"{emoji}  {verdict}\n\nConfidence : {confidence:.1f}%"
    return result, confidence, fake_prob

with gr.Blocks(
    title="Deepfake Audio Detector",
    theme=gr.themes.Soft(primary_hue="red", neutral_hue="slate")
) as demo:

    gr.Markdown("""
    # 🎙️ Deepfake Audio Detection System
    Upload a **.wav** file to detect whether it is real or AI-generated speech.
    """)

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Upload audio (.wav)"
            )
            submit_btn = gr.Button("🔍 Analyze", variant="primary")
        with gr.Column():
            verdict_out    = gr.Textbox(label="Prediction",
                                        lines=3, interactive=False)
            confidence_out = gr.Number(label="Confidence (%)",
                                       interactive=False)
            fake_prob_out  = gr.Number(label="Fake probability (%)",
                                       interactive=False)

    submit_btn.click(
        fn      = gradio_predict,
        inputs  = [audio_input],
        outputs = [verdict_out, confidence_out, fake_prob_out]
    )
    gr.Markdown("""
    ---
    - Fake probability **above 50%** → **Fake**
    - Fake probability **below 50%** → **Real**
    """)

demo.launch(
    share         = True,
    debug         = False,   # ← must be False or cell hangs
    quiet         = True,    # ← suppresses log spam
    show_error    = True     # ← shows errors in UI instead of crashing
)

---
## PHASE 6 — Performance Optimization Notes

All optimizations are already baked into the pipeline above:

| Technique | Where used |
|---|---|
| **Parallel feature extraction** | `joblib.Parallel` with `prefer='threads'` in Cell 2.3 |
| **Progress bar** | `tqdm.auto` wraps the file iterator |
| **Corrupt-file handling** | `extract_features` returns `None`; pairs are filtered out |
| **GPU acceleration** | XGBoost `device='cuda'` (Colab T4/A100) |
| **Balanced sampling** | `groupby.apply.sample` ensures equal class counts |
| **Data-leakage prevention** | Scaler fit on train only, applied to test |
| **Disk caching** | `X.npy` / `y.npy` saved so Phase 2 can be skipped on re-runs |

---
## PHASE 7 — Quick Self-Test (Optional)
Run a small smoke test to verify the full pipeline end-to-end.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7.1 — Smoke Test: predict on 10 held-out samples
# ─────────────────────────────────────────────────────────────
import random

test_samples = df_subset.sample(10, random_state=99)

print("🔬 Smoke test on 10 random samples:\n")
print(f"{'File':<55} {'True':>5} {'Pred':>5} {'Conf%':>7}")
print("-" * 75)

for _, row in test_samples.iterrows():
    label_str, conf = predict_audio(row["file_path"])
    true_str  = "REAL" if row["label"] == 0 else "FAKE"
    pred_str  = "REAL" if "REAL" in label_str else "FAKE"
    match     = "✅" if true_str == pred_str else "❌"
    fname     = os.path.basename(row["file_path"])
    print(f"{match} {fname:<52} {true_str:>5} {pred_str:>5} {conf:>6.1f}%")

print("\n✅ Smoke test complete.")